# Mini FSD training on Colab

Thin by design (ADR 0007): this notebook mounts Drive, installs the repo at a
pinned commit, unpacks the data and calls a `minifsd.train` module. All the
logic is in the repo, tested locally.

Before the first run, upload the packs from `data/packs/` (made by
`python -m minifsd.train.pack`) to `My Drive/mini-fsd/packs/`.

Runtime > Change runtime type > T4 GPU. If the session drops, run all cells
again: training resumes from the last checkpoint on Drive.

In [36]:
# Settings for this run. COMMIT pins the code, so a result can be traced
# back to it; use the commit you pushed last (git rev-parse --short HEAD).
COMMIT = "main"
PACKS = ["v9.zip", "coco-people.zip", "v10.zip"]
DRIVE = "/content/drive/MyDrive/mini-fsd"
RUN = "e2a-segformer-b0"  # checkpoints and logs go to DRIVE/runs/RUN

In [37]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [38]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
# The repo at the pinned commit. Extra packages, pinned, are added here per
# experiment and recorded in ADR 0007.
# --force-reinstall: a new COMMIT must replace the old install even though
# the version number is the same. --no-deps: keep Colab's numpy and scipy.
!pip install -q --force-reinstall --no-deps "git+https://github.com/SegunAdewola/mini-fsd@{COMMIT}"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [41]:
import pathlib
import shutil

from minifsd.train import pack

data = pathlib.Path("/content/data")
data.mkdir(exist_ok=True)
for name in PACKS:
    local = data / name
    if not local.exists():
        shutil.copy(f"{DRIVE}/packs/{name}", local)
    folder = data / name.removesuffix(".zip")
    if not folder.exists():
        print("unpacked", pack.unpack(local, data))

## E2a: SegFormer-B0 floor segmentation

Smoke run first: 64 frames, one epoch, on the session disk. It checks the
whole loop on the GPU in a minute or two before the real run.

In [10]:
!python -m minifsd.train.segformer --train /content/data/v9/train --val /content/data/v9/val --run-dir /content/smoke --epochs 1 --limit 64

config.json: 100% 70.0k/70.0k [00:00<00:00, 78.5MB/s]
[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.

pytorch_model.bin: downloading bytes:  86% 12.4M/14.4M [00:00<00:00, 19.7MB/s]
pytorch_model.bin: downloading bytes: 100% 13.5M/13.5M [00:00<00:00, 19.0MB/s, 1.32MB/s  ]
pytorch_model.bin: reconstructing file: 100% 14.4M/14.4M [00:00<00:00, 20.2MB/s, 1.41MB/s  ]
Loading weights: 100% 192/192 [00:00<00:00, 23814.36it/s]
[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.weight                                       | UNEXPECTED | 
classifier.bias                                         | UNEXPECTED | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_he

The real run: all 9,000 v9 train frames, 20 epochs, checkpoints on Drive.
If the session drops, run every cell again from the top: it resumes after
the last finished epoch.

In [11]:
!python -m minifsd.train.segformer --train /content/data/v9/train --val /content/data/v9/val --run-dir {DRIVE}/runs/{RUN} --epochs 20

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100% 192/192 [00:00<00:00, 23804.50it/s]
[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.weight                                       | UNEXPECTED | 
classifier.bias                                         | UNEXPECTED | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.classifier.bias            

In [12]:
# One line per finished epoch; best/ holds the highest val IoU so far.
!cat {DRIVE}/runs/{RUN}/log.jsonl

{"epoch": 0, "train_loss": 0.05215786155561606, "val_iou": 0.9768741412862171, "best_iou": 0.9768741412862171, "seconds": 82.0, "device": "Tesla T4"}
{"epoch": 1, "train_loss": 0.02140828567246596, "val_iou": 0.9840429411680561, "best_iou": 0.9840429411680561, "seconds": 73.3, "device": "Tesla T4"}
{"epoch": 2, "train_loss": 0.015279178099499809, "val_iou": 0.9852410425410074, "best_iou": 0.9852410425410074, "seconds": 74.3, "device": "Tesla T4"}
{"epoch": 3, "train_loss": 0.014534363969332642, "val_iou": 0.9873176543044618, "best_iou": 0.9873176543044618, "seconds": 74.1, "device": "Tesla T4"}
{"epoch": 4, "train_loss": 0.011765132310075893, "val_iou": 0.9881484839473546, "best_iou": 0.9881484839473546, "seconds": 74.9, "device": "Tesla T4"}
{"epoch": 5, "train_loss": 0.010633069614155425, "val_iou": 0.9904128094578222, "best_iou": 0.9904128094578222, "seconds": 74.9, "device": "Tesla T4"}
{"epoch": 6, "train_loss": 0.01096650639259153, "val_iou": 0.9886441483214091, "best_iou": 0.990

## E2b: frozen DINOv2 ViT-S/14 with a floor head

Same data and loop as E2a; only the head trains. Smoke run first, then the
real run (10 epochs). Run the settings, Drive, install and unpack cells
first if this is a new session.

In [16]:
!python -m minifsd.train.dino --train /content/data/v9/train --val /content/data/v9/val --run-dir /content/smoke-dino --epochs 1 --limit 64

config.json: 100% 547/547 [00:00<00:00, 2.36MB/s]

model.safetensors: downloading bytes:  84% 74.0M/88.2M [00:00<00:00, 128MB/s, 4.74MB/s  ] 
model.safetensors: downloading bytes: 100% 83.7M/83.7M [00:00<00:00, 91.9MB/s, 8.10MB/s  ]
model.safetensors: reconstructing file: 100% 88.2M/88.2M [00:00<00:00, 96.9MB/s, 8.64MB/s  ]
Loading weights: 100% 223/223 [00:00<00:00, 4948.36it/s]
epoch 0 step 0/2 loss 0.8439
Writing model shards: 100% 1/1 [00:00<00:00,  7.63it/s]
{"epoch": 0, "train_loss": 0.7386539280414581, "val_iou": 0.47675304786919875, "best_iou": 0.47675304786919875, "seconds": 7.1, "device": "Tesla T4"}


In [17]:
!python -m minifsd.train.dino --train /content/data/v9/train --val /content/data/v9/val --run-dir {DRIVE}/runs/e2b-dinov2-s

Loading weights: 100% 223/223 [00:00<00:00, 4744.21it/s]
epoch 0 step 0/282 loss 0.8654
epoch 0 step 100/282 loss 0.0542
epoch 0 step 200/282 loss 0.0492
Writing model shards: 100% 1/1 [00:00<00:00,  3.73it/s]
{"epoch": 0, "train_loss": 0.07715453000863393, "val_iou": 0.9475390970630674, "best_iou": 0.9475390970630674, "seconds": 63.9, "device": "Tesla T4"}
epoch 1 step 0/282 loss 0.0500
epoch 1 step 100/282 loss 0.0451
epoch 1 step 200/282 loss 0.0390
Writing model shards: 100% 1/1 [00:00<00:00,  4.24it/s]
{"epoch": 1, "train_loss": 0.04695285337169965, "val_iou": 0.951632105349861, "best_iou": 0.951632105349861, "seconds": 59.9, "device": "Tesla T4"}
epoch 2 step 0/282 loss 0.0441
epoch 2 step 100/282 loss 0.0428
epoch 2 step 200/282 loss 0.0423
Writing model shards: 100% 1/1 [00:00<00:00,  4.57it/s]
{"epoch": 2, "train_loss": 0.04341928028729227, "val_iou": 0.9531209878166091, "best_iou": 0.9531209878166091, "seconds": 60.5, "device": "Tesla T4"}
epoch 3 step 0/282 loss 0.0356
epoch

In [18]:
!cat {DRIVE}/runs/e2b-dinov2-s/log.jsonl

{"epoch": 0, "train_loss": 0.07715453000863393, "val_iou": 0.9475390970630674, "best_iou": 0.9475390970630674, "seconds": 63.9, "device": "Tesla T4"}
{"epoch": 1, "train_loss": 0.04695285337169965, "val_iou": 0.951632105349861, "best_iou": 0.951632105349861, "seconds": 59.9, "device": "Tesla T4"}
{"epoch": 2, "train_loss": 0.04341928028729227, "val_iou": 0.9531209878166091, "best_iou": 0.9531209878166091, "seconds": 60.5, "device": "Tesla T4"}
{"epoch": 3, "train_loss": 0.04156620032257504, "val_iou": 0.954332267878117, "best_iou": 0.954332267878117, "seconds": 61.0, "device": "Tesla T4"}
{"epoch": 4, "train_loss": 0.04093136115868886, "val_iou": 0.9550105269777807, "best_iou": 0.9550105269777807, "seconds": 61.0, "device": "Tesla T4"}
{"epoch": 5, "train_loss": 0.039966086943944296, "val_iou": 0.9555099225158602, "best_iou": 0.9555099225158602, "seconds": 60.9, "device": "Tesla T4"}
{"epoch": 6, "train_loss": 0.039414391958051256, "val_iou": 0.9552632876562013, "best_iou": 0.955509922

## E2c: both floor models retrained on v10 (furniture)

Same recipes as E2a and E2b, new data. Run the settings and unpack cells
first so v10 is on the session disk. DINOv2 first: it is the quicker run.

In [ ]:
!python -m minifsd.train.dino --train /content/data/v10/train --val /content/data/v10/val --run-dir {DRIVE}/runs/e2c-dinov2-s-v10

In [28]:
!python -m minifsd.train.segformer --train /content/data/v10/train --val /content/data/v10/val --run-dir {DRIVE}/runs/e2c-segformer-b0-v10

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100% 192/192 [00:00<00:00, 25346.42it/s]
[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.weight                                       | UNEXPECTED | 
classifier.bias                                         | UNEXPECTED | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.classifier.bias                             | MISSING    | 
decode_head.batch_norm.weight                           | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.batch_norm.num_batches_trac

In [30]:
!tail -n 3 {DRIVE}/runs/e2c-dinov2-s-v10/log.jsonl {DRIVE}/runs/e2c-segformer-b0-v10/log.jsonl

==> /content/drive/MyDrive/mini-fsd/runs/e2c-dinov2-s-v10/log.jsonl <==
{"epoch": 7, "train_loss": 0.04394377242856556, "val_iou": 0.9518877860583831, "best_iou": 0.9518877860583831, "seconds": 61.2, "device": "Tesla T4"}
{"epoch": 8, "train_loss": 0.04356642346249686, "val_iou": 0.9518526228231464, "best_iou": 0.9518877860583831, "seconds": 63.3, "device": "Tesla T4"}
{"epoch": 9, "train_loss": 0.043035225619872414, "val_iou": 0.9514483185369359, "best_iou": 0.9518877860583831, "seconds": 61.4, "device": "Tesla T4"}

==> /content/drive/MyDrive/mini-fsd/runs/e2c-segformer-b0-v10/log.jsonl <==
{"epoch": 17, "train_loss": 0.007554476245823834, "val_iou": 0.9906494648956845, "best_iou": 0.9906494648956845, "seconds": 74.6, "device": "Tesla T4"}
{"epoch": 18, "train_loss": 0.007420569846199618, "val_iou": 0.9900420072442133, "best_iou": 0.9906494648956845, "seconds": 74.1, "device": "Tesla T4"}
{"epoch": 19, "train_loss": 0.00720915518494116, "val_iou": 0.9900762288542256, "best_iou": 0.99

## E1a: YOLO26n on v10 plus COCO people

ultralytics is AGPL-3.0 and lives on Colab only (ADR 0008), pinned here.
Smoke run first: one epoch on v10 val, which also checks the ONNX export.
Run after E2c: one GPU, one job at a time.

In [42]:
!pip install -q "ultralytics==8.4.214"

In [43]:
!python -m minifsd.train.yolo --train v10=/content/data/v10/val --val v10=/content/data/v10/val --run-dir /content/smoke-yolo/run --work-dir /content/smoke-yolo/data --epochs 1

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
New https://pypi.org/project/ultralytics/8.4.219 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.214 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/smoke-yolo/data/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=

In [ ]:
!python -m minifsd.train.yolo --train v10=/content/data/v10/train coco=/content/data/coco-people/train --val v10=/content/data/v10/val coco=/content/data/coco-people/val --run-dir {DRIVE}/runs/e1a-yolo26n

New https://pypi.org/project/ultralytics/8.4.219 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.214 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, 

In [ ]:
!tail -n 3 {DRIVE}/runs/e1a-yolo26n/results.csv

## E1b: RF-DETR Nano on v10 plus COCO people

RF-DETR (Apache-2.0) is installed on Colab only, pinned here, like ultralytics.
It uses the same data and the same 320 px input as E1a. Smoke run first: one
epoch on v10 val, which also checks the ONNX export. Run after E1a, since there
is one GPU and one job at a time.

In [ ]:
!pip install -q "rfdetr[train]==1.12.1"

In [ ]:
!python -m minifsd.train.rf_detr --train v10=/content/data/v10/val --val v10=/content/data/v10/val --run-dir /content/smoke-rfdetr/run --work-dir /content/smoke-rfdetr/data --epochs 1

In [ ]:
!python -m minifsd.train.rf_detr --train v10=/content/data/v10/train coco=/content/data/coco-people/train --val v10=/content/data/v10/val coco=/content/data/coco-people/val --run-dir {DRIVE}/runs/e1b-rfdetr-nano

## E3a: Depth Anything 3 Small, zero-shot, in its own environment

DA3's package needs numpy<2 and loads 3D tools on import, so it gets a
throwaway virtual environment. It predicts depth for the images exported on
the Mac; the harness scores them there.

In [ ]:
!pip install -q virtualenv && virtualenv -q /content/da3env
!/content/da3env/bin/pip install -q "depth-anything-3==0.1.1"
!/content/da3env/bin/pip install -q --no-deps "git+https://github.com/SegunAdewola/mini-fsd@{COMMIT}"

In [ ]:
shutil.copy(f"{DRIVE}/packs/da3-inputs.zip", "/content/data/da3-inputs.zip")
print(pack.unpack(pathlib.Path("/content/data/da3-inputs.zip"), pathlib.Path("/content/data")))

In [ ]:
!mkdir -p {DRIVE}/models
!/content/da3env/bin/python -m minifsd.train.da3_predict --images /content/data/da3-inputs --out {DRIVE}/models/da3-small-predictions.npz